In [1]:
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score

from data_utils import load_data_for_n, get_fold_split, N_LEVELS

MODELS_DIR = Path("../Models/FFNN_R5")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
class DepressionClassifier(nn.Module):
    def __init__(self, input_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 64),  nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

In [3]:
def train_model(X_train, y_train, seed, epochs=30, batch_size=32, lr=0.001):
    """
    Train one FFNN. The seed controls weight initialization AND per-epoch
    shuffling order, so setting it makes the run reproducible.
    """
    # Set all sources of randomness
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = DepressionClassifier(input_dim=X_train.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

    n_samples = X_tensor.shape[0]

    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(n_samples)   # uses the manual_seed above

        for i in range(0, n_samples, batch_size):
            indices = permutation[i:i + batch_size]
            batch_X, batch_y = X_tensor[indices], y_tensor[indices]

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    return model

In [4]:
def evaluate_model(model, X_test, y_test, speaker_test):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    with torch.no_grad():
        outputs = model(X_tensor).numpy().flatten()

    seg_preds = (outputs > 0.5).astype(int)
    seg_acc = (seg_preds == y_test).mean()
    seg_f1 = f1_score(y_test, seg_preds)

    rec_true, rec_pred = [], []
    for spk in np.unique(speaker_test):
        mask = (speaker_test == spk)
        rec_true.append(y_test[mask][0])
        rec_pred.append(int(seg_preds[mask].mean() > 0.5))

    rec_true, rec_pred = np.array(rec_true), np.array(rec_pred)
    rec_acc = (rec_pred == rec_true).mean()
    rec_f1 = f1_score(rec_true, rec_pred)

    return seg_acc, seg_f1, rec_acc, rec_f1

In [5]:
def run_cv_ffnn_r5(n: int, seeds=(0, 1, 2, 3, 4), epochs=30):
    """
    5-fold CV × R random-init repetitions.
    For each fold, train R separate models with different seeds,
    then average their per-fold results.

    This matches the paper's protocol (R=5).
    """
    X, y, fold, speaker = load_data_for_n(n)

    # Store per-fold-per-seed results
    rec_accs_per_fold = []
    rec_f1s_per_fold = []
    seg_accs_per_fold = []
    seg_f1s_per_fold = []

    for fold_number in [1, 2, 3, 4, 5]:
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(
            X, y, fold, speaker, fold_number
        )

        fold_rec_accs, fold_rec_f1s = [], []
        fold_seg_accs, fold_seg_f1s = [], []

        for seed in seeds:
            model = train_model(X_train, y_train, seed=seed, epochs=epochs)
            s_acc, s_f1, r_acc, r_f1 = evaluate_model(model, X_test, y_test, speaker_test)
            fold_seg_accs.append(s_acc); fold_seg_f1s.append(s_f1)
            fold_rec_accs.append(r_acc); fold_rec_f1s.append(r_f1)

        # For each fold, average across the R seeds (this matches the paper)
        rec_accs_per_fold.append(np.mean(fold_rec_accs))
        rec_f1s_per_fold.append(np.mean(fold_rec_f1s))
        seg_accs_per_fold.append(np.mean(fold_seg_accs))
        seg_f1s_per_fold.append(np.mean(fold_seg_f1s))

    return {
        "seg_acc_mean": np.mean(seg_accs_per_fold), "seg_acc_std": np.std(seg_accs_per_fold),
        "seg_f1_mean":  np.mean(seg_f1s_per_fold),  "seg_f1_std":  np.std(seg_f1s_per_fold),
        "rec_acc_mean": np.mean(rec_accs_per_fold), "rec_acc_std": np.std(rec_accs_per_fold),
        "rec_f1_mean":  np.mean(rec_f1s_per_fold),  "rec_f1_std":  np.std(rec_f1s_per_fold),
    }

In [6]:
results_r5 = {}

for n in tqdm(N_LEVELS, desc="N-levels"):
    print(f"\n=== N={n} ===")
    results_r5[n] = run_cv_ffnn_r5(n=n)
    r = results_r5[n]
    print(f"  Rec: Acc={r['rec_acc_mean']:.4f}±{r['rec_acc_std']:.4f}, "
          f"F1={r['rec_f1_mean']:.4f}±{r['rec_f1_std']:.4f}")

# Save
with open(MODELS_DIR / "results_summary.json", "w") as f:
    json.dump({str(n): r for n, r in results_r5.items()}, f, indent=2)

print("\nDone. Results saved to results_summary.json")

N-levels:   0%|          | 0/7 [00:00<?, ?it/s]


=== N=1 ===


N-levels:  14%|█▍        | 1/7 [00:01<00:08,  1.45s/it]

  Rec: Acc=0.8793±0.0749, F1=0.8959±0.0545

=== N=2 ===


N-levels:  29%|██▊       | 2/7 [00:03<00:07,  1.54s/it]

...

  Rec: Acc=0.8627±0.1044, F1=0.8763±0.0837

=== N=32 ===


N-levels:  86%|████████▌ | 6/7 [00:45<00:11, 11.74s/it]

  Rec: Acc=0.8175±0.0762, F1=0.8262±0.0545

=== N=64 ===


N-levels: 100%|██████████| 7/7 [01:29<00:00, 12.80s/it]

  Rec: Acc=0.8178±0.0889, F1=0.8272±0.0706

Done. Results saved to results_summary.json


In [7]:
def run_cv_ffnn_r5_folds(n: int, seeds=(0,1,2,3,4), epochs=30):
    """Same as run_cv_ffnn_r5 but ALSO returns the 5 per-fold F1s,
    which are needed for paired significance testing."""
    X, y, fold, speaker = load_data_for_n(n)
    rec_f1s_per_fold, rec_accs_per_fold = [], []

    for fold_number in [1, 2, 3, 4, 5]:
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(
            X, y, fold, speaker, fold_number
        )
        fold_f1s, fold_accs = [], []
        for seed in seeds:
            model = train_model(X_train, y_train, seed=seed, epochs=epochs)
            _, _, r_acc, r_f1 = evaluate_model(model, X_test, y_test, speaker_test)
            fold_f1s.append(r_f1); fold_accs.append(r_acc)

        rec_f1s_per_fold.append(np.mean(fold_f1s))
        rec_accs_per_fold.append(np.mean(fold_accs))

    return {
        "f1_mean": np.mean(rec_f1s_per_fold), "f1_std": np.std(rec_f1s_per_fold),
        "acc_mean": np.mean(rec_accs_per_fold), "acc_std": np.std(rec_accs_per_fold),
        "fold_f1s": rec_f1s_per_fold, "fold_accs": rec_accs_per_fold,
    }


results_sil_folds = {}
for n in tqdm(N_LEVELS, desc="FFNN-SIL"):
    results_sil_folds[n] = run_cv_ffnn_r5_folds(n)
    print(f"N={n:>2}: F1={results_sil_folds[n]['f1_mean']*100:.2f}% "
          f"± {results_sil_folds[n]['f1_std']*100:.2f}%")

FFNN-SIL:  14%|█▍        | 1/7 [00:00<00:04,  1.22it/s]

N= 1: F1=89.59% ± 5.45%


FFNN-SIL:  29%|██▊       | 2/7 [00:02<00:06,  1.21s/it]

N= 2: F1=89.32% ± 4.06%


FFNN-SIL:  43%|████▎     | 3/7 [00:05<00:07,  1.98s/it]

...

N=16: F1=87.63% ± 8.37%


FFNN-SIL:  86%|████████▌ | 6/7 [00:43<00:11, 11.31s/it]

N=32: F1=82.62% ± 5.45%


FFNN-SIL: 100%|██████████| 7/7 [01:27<00:00, 12.44s/it]

N=64: F1=82.72% ± 7.06%


In [9]:
import json

with open("../Models/MIL/results_minn_pooled.json") as f:
    _loaded = json.load(f)

# convert string keys back to ints
results_minn = {pool: {int(n): r for n, r in res.items()}
                for pool, res in _loaded.items()}

print("Loaded MINN:", list(results_minn.keys()),
      "| N-levels:", sorted(results_minn["max"].keys()))

Loaded MINN: ['max', 'mean'] | N-levels: [1, 2, 4, 8, 16, 32, 64]


In [10]:
from scipy import stats

print("FFNN-SIL (majority vote)  vs  MINN-max (pooling)")
print(f"{'N':>4} | {'SIL F1':>8} | {'MIL F1':>8} | {'diff':>7} | {'p':>8} | sig")
print("-" * 56)

for n in N_LEVELS:
    sil = np.array(results_sil_folds[n]["fold_f1s"])
    mil = np.array(results_minn["max"][n]["fold_f1s"])
    diff = mil.mean() - sil.mean()

    t_stat, p_val = stats.ttest_rel(mil, sil)     # paired, two-sided

    sig = "**" if p_val < 0.0071 else ("*" if p_val < 0.05 else "")
    print(f"{n:>4} | {sil.mean()*100:7.2f}% | {mil.mean()*100:7.2f}% "
          f"| {diff*100:+6.2f} | {p_val:8.4f} | {sig}")

print("\n*  p < 0.05 (uncorrected)")
print("** p < 0.0071 (Bonferroni-corrected for 7 comparisons)")

FFNN-SIL (majority vote)  vs  MINN-max (pooling)
   N |   SIL F1 |   MIL F1 |    diff |        p | sig
--------------------------------------------------------
   1 |   89.59% |   90.89% |  +1.30 |   0.3651 | 
   2 |   89.32% |   90.89% |  +1.57 |   0.1667 | 
   4 |   90.51% |   91.33% |  +0.82 |   0.5064 | 
   8 |   90.81% |   88.86% |  -1.95 |   0.1861 | 
  16 |   87.63% |   90.38% |  +2.75 |   0.4371 | 
  32 |   82.62% |   89.34% |  +6.72 |   0.0058 | **
  64 |   82.72% |   89.34% |  +6.61 |   0.0571 | 

*  p < 0.05 (uncorrected)
** p < 0.0071 (Bonferroni-corrected for 7 comparisons)


In [11]:
import json
from pathlib import Path

Path("../Models/FFNN_R5").mkdir(parents=True, exist_ok=True)

def _clean(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, list):
            out[k] = [float(x) for x in v]
        else:
            out[k] = float(v)
    return out

with open("../Models/FFNN_R5/results_sil_folds.json", "w") as f:
    json.dump({str(n): _clean(r) for n, r in results_sil_folds.items()}, f, indent=2)

print("Saved FFNN-SIL with per-fold F1s")
print("N-levels:", sorted(results_sil_folds.keys()))

Saved FFNN-SIL with per-fold F1s
N-levels: [1, 2, 4, 8, 16, 32, 64]
